# Face-anchored gesture challenge — offline pipeline (Colab T4)
Runtime → Change runtime type → **T4 GPU**. Run top to bottom. Everything is saved under `/content/drive/MyDrive/csbp441` so a disconnect does not lose results.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/csbp441'
!mkdir -p $WORK

In [ ]:
REPO = 'https://github.com/AbdulwahhabAli/Face-Anchored-Gesture-Challenge-for-Rotating-QR-Classroom-Attendance.git'
!git clone -q $REPO /content/repo || (cd /content/repo && git pull -q)
%cd /content/repo
!pip -q install -r requirements.txt && pip -q install -e .
!python -m pytest -q

## 1. HaGRID subset (6 classes, split by subject)
Get the download links for the annotations and the per-class image archives (`like, peace, palm, fist, ok, one`) from the HaGRID GitHub README (https://github.com/hukenovs/hagrid) and paste them below. The 512 px version is enough for a first run; full resolution gives the ROI more pixels to work with.

In [ ]:
ANN_URL = ''   # annotations zip
IMG_URLS = {   # class -> images zip
    'like': '', 'peace': '', 'palm': '', 'fist': '', 'ok': '', 'one': '',
}
import os
os.makedirs('/content/hagrid/images', exist_ok=True)
if ANN_URL:
    !wget -q -O /content/hagrid/ann.zip "$ANN_URL" && unzip -q -o /content/hagrid/ann.zip -d /content/hagrid/annotations
for c, u in IMG_URLS.items():
    if u:
        !wget -q -O /content/hagrid/{c}.zip "{u}" && unzip -q -o /content/hagrid/{c}.zip -d /content/hagrid/images && rm /content/hagrid/{c}.zip

In [ ]:
!python scripts/prepare_hagrid.py --ann-dir /content/hagrid/annotations --images-root /content/hagrid/images \
    --out $WORK/hagrid_yolo --per-class 400 --seed 441

## 2. Fine-tune YOLOv8n (~30 epochs, < 1 h on T4)

In [ ]:
!python scripts/train_yolo.py --data $WORK/hagrid_yolo --epochs 30 --device 0 --project $WORK/runs/train
!cp models/gesture_yolov8n.pt $WORK/gesture_yolov8n.pt

## 3. Baseline vs ROI on the HaGRID test split

In [ ]:
!python scripts/predict.py --dataset hagrid --root $WORK/hagrid_yolo --out $WORK/runs/preds/hagrid \
    --configs full roi_s2_up roi_s3_up roi_s4_up roi_s3_noup roi_s3_up_scrfd --set models.device=0
!python scripts/evaluate.py --preds $WORK/runs/preds/hagrid --out $WORK/runs/eval/hagrid

## 4. Classroom (volunteer) set
Upload `data/classroom` (images + `labels.csv`, collected with the app's `/collect` page) to `$WORK/classroom`.

In [ ]:
!python scripts/predict.py --dataset classroom --root $WORK/classroom --out $WORK/runs/preds/classroom \
    --configs full roi_s2_up roi_s3_up roi_s4_up roi_s3_noup roi_s3_up_scrfd --set models.device=0
!python scripts/evaluate.py --preds $WORK/runs/preds/classroom --out $WORK/runs/eval/classroom

## 5. Face threshold (LFW) and end-to-end FRR / FAR
Put the LFW images (`lfw/<Name>/<Name>_0001.jpg`) and `pairsDevTrain.txt`, `pairsDevTest.txt` in `$WORK/lfw`.

In [ ]:
!python scripts/eval_face.py --lfw-root $WORK/lfw/lfw --lfw-pairs $WORK/lfw/pairsDevTrain.txt \
    --lfw-test-pairs $WORK/lfw/pairsDevTest.txt --classroom-emb $WORK/runs/preds/classroom/embeddings.npz --out $WORK/runs/face
!python scripts/eval_e2e.py --preds $WORK/runs/preds/classroom --threshold $WORK/runs/face/threshold.json --out $WORK/runs/eval/classroom

## 6. Latency on the T4 and figures

In [ ]:
!python scripts/bench_latency.py --images-dir $WORK/classroom/images --device 0 --configs full roi_s3_up roi_s3_up_scrfd --out $WORK/runs/latency
!python scripts/make_figures.py --preds $WORK/runs/preds/classroom --eval $WORK/runs/eval/classroom \
    --images-root $WORK/classroom --latency $WORK/runs/latency/latency_0.csv --out $WORK/runs/figures/classroom
from IPython.display import Markdown, Image, display
display(Markdown(open(f'{WORK}/runs/eval/classroom/report.md').read()))
display(Image(f'{WORK}/runs/figures/classroom/acc_by_condition.png'))